In [1]:
import pandas as pd 
import numpy as np 

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

deberta_path = "microsoft/deberta-v3-small"
roberta_path = "roberta-base"

id2label = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
label2id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_path)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_path)

deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_path, num_labels=5).to(device)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_path, num_labels=5).to(device)

deberta_model.eval()
roberta_model.eval()

Using device: cuda


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

In [4]:
import torch.nn.functional as F

prompt = train.loc[25, "prompt"]
inputs = deberta_tokenizer(prompt, return_tensors="pt").to(deberta_model.device)

with torch.no_grad():
    logits = deberta_model(**inputs).logits
    probs = F.softmax(logits, dim=-1).squeeze()

max_idx = torch.argmax(probs).item()
max_prob = probs[max_idx].item()

print(f"{id2label[max_idx]}, {max_prob}")

C, 0.255615234375


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [5]:
prompt = train.loc[25, "prompt"]
deberta_inputs = deberta_tokenizer(prompt, return_tensors="pt").to(deberta_model.device)
with torch.no_grad():
    deberta_logits = deberta_model(**deberta_inputs).logits
    deberta_probs = F.softmax(deberta_logits, dim=-1).squeeze()

roberta_inputs = roberta_tokenizer(prompt, return_tensors="pt").to(roberta_model.device)
with torch.no_grad():
    roberta_logits = roberta_model(**roberta_inputs).logits
    roberta_probs = F.softmax(roberta_logits, dim=-1).squeeze()

avg_probs = (deberta_probs + roberta_probs) / 2

max_idx = torch.argmax(avg_probs).item()

print(f"{id2label[max_idx]}")

C


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?


In [6]:
weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

max_idx = torch.argmax(weighted_probs).item()

print(f"{id2label[max_idx]}")

C


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

Example : C A E

In [7]:
ranked_indices = torch.argsort(weighted_probs, descending=True)
top_3_labels = [id2label[idx.item()] for idx in ranked_indices[:3]]

prediction_string = " ".join(top_3_labels)

print(prediction_string)

C E A


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [8]:
predictions = []
for index, row in test.iterrows():
    prompt = row["prompt"]
    
    deberta_inputs = deberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(deberta_model.device)
    with torch.no_grad():
        deberta_logits = deberta_model(**deberta_inputs).logits
        deberta_probs = F.softmax(deberta_logits, dim=-1).squeeze()

    roberta_inputs = roberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(roberta_model.device)
    with torch.no_grad():
        roberta_logits = roberta_model(**roberta_inputs).logits
        roberta_probs = F.softmax(roberta_logits, dim=-1).squeeze()

    weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    ranked_indices = torch.argsort(weighted_probs, descending=True)
    top_3_labels = [id2label[idx.item()] for idx in ranked_indices[:3]]
    prediction_string = " ".join(top_3_labels)
    
    predictions.append({"id": row["id"], "prediction": prediction_string})

submission_df = pd.DataFrame(predictions)
submission_df.to_csv("submission.csv", index=False)

num_rows = len(submission_df)
print(num_rows)

500


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [9]:
differences_count = 0
instruction = "Answer the following multiple-choice question carefully: "

for index, row in test.head(50).iterrows():
    original_prompt = row["prompt"]
    aug_prompt = instruction + original_prompt
    
    orig_inputs = deberta_tokenizer(original_prompt, return_tensors="pt", truncation=True, max_length=512).to(deberta_model.device)
    with torch.no_grad():
        orig_logits = deberta_model(**orig_inputs).logits
        orig_probs = F.softmax(orig_logits, dim=-1).squeeze()
    
    orig_top1 = torch.argmax(orig_probs).item()
    
    aug_inputs = deberta_tokenizer(aug_prompt, return_tensors="pt", truncation=True, max_length=512).to(deberta_model.device)
    with torch.no_grad():
        aug_logits = deberta_model(**aug_inputs).logits
        aug_probs = F.softmax(aug_logits, dim=-1).squeeze()
        
    tta_probs = (orig_probs + aug_probs) / 2
    
    tta_top1 = torch.argmax(tta_probs).item()
    
    if orig_top1 != tta_top1:
        differences_count += 1

print(differences_count)

5


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?

In [10]:
differences_count = 0

for index, row in test.head(100).iterrows():
    prompt = row["prompt"]
    
    deberta_inputs = deberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(deberta_model.device)
    with torch.no_grad():
        deberta_logits = deberta_model(**deberta_inputs).logits
        deberta_probs = F.softmax(deberta_logits, dim=-1).squeeze()
        
    deberta_top1 = torch.argmax(deberta_probs).item()
    
    roberta_inputs = roberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(roberta_model.device)
    with torch.no_grad():
        roberta_logits = roberta_model(**roberta_inputs).logits
        roberta_probs = F.softmax(roberta_logits, dim=-1).squeeze()
        
    weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
    
    ensemble_top1 = torch.argmax(weighted_probs).item()
    
    if deberta_top1 != ensemble_top1:
        differences_count += 1

print(differences_count)

14


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?


In [11]:
positive_gain_count = 0

for index, row in test.head(100).iterrows():
    prompt = row["prompt"]
    
    deberta_inputs = deberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(deberta_model.device)
    with torch.no_grad():
        deberta_logits = deberta_model(**deberta_inputs).logits
        deberta_probs = F.softmax(deberta_logits, dim=-1).squeeze()
    deberta_confidence = torch.max(deberta_probs).item()
    
    roberta_inputs = roberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(roberta_model.device)
    with torch.no_grad():
        roberta_logits = roberta_model(**roberta_inputs).logits
        roberta_probs = F.softmax(roberta_logits, dim=-1).squeeze()
        
    weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
    ensemble_confidence = torch.max(weighted_probs).item()
    
    confidence_gain = ensemble_confidence - deberta_confidence
    
    if confidence_gain > 0:
        positive_gain_count += 1

print(positive_gain_count)

1


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

*
Examples:

A C D vs. A D C

In [12]:
changed_ranking_count = 0

for index, row in test.head(100).iterrows():
    prompt = row["prompt"]
    
    deberta_inputs = deberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(deberta_model.device)
    with torch.no_grad():
        deberta_logits = deberta_model(**deberta_inputs).logits
        deberta_probs = F.softmax(deberta_logits, dim=-1).squeeze()
        
    deberta_ranked_indices = torch.argsort(deberta_probs, descending=True)
    deberta_top3_labels = [id2label[idx.item()] for idx in deberta_ranked_indices[:3]]
    deberta_top3_str = " ".join(deberta_top3_labels)
    
    roberta_inputs = roberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(roberta_model.device)
    with torch.no_grad():
        roberta_logits = roberta_model(**roberta_inputs).logits
        roberta_probs = F.softmax(roberta_logits, dim=-1).squeeze()
        
    weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
    
    ensemble_ranked_indices = torch.argsort(weighted_probs, descending=True)
    ensemble_top3_labels = [id2label[idx.item()] for idx in ensemble_ranked_indices[:3]]
    ensemble_top3_str = " ".join(ensemble_top3_labels)
    
    if deberta_top3_str != ensemble_top3_str:
        changed_ranking_count += 1

print(changed_ranking_count)

29


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score? 

*
(Round to 4 decimal places.)


In [13]:
def calculate_ap3(actual, predicted):
    for i, pred in enumerate(predicted[:3]):
        if pred == actual:
            return 1.0 / (i + 1)
    return 0.0

ap_scores = []

for index, row in train.head(100).iterrows():
    prompt = row["prompt"]
    actual_answer = row["answer"]  
    
    deberta_inputs = deberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(deberta_model.device)
    with torch.no_grad():
        deberta_logits = deberta_model(**deberta_inputs).logits
        deberta_probs = F.softmax(deberta_logits, dim=-1).squeeze()
        
    roberta_inputs = roberta_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(roberta_model.device)
    with torch.no_grad():
        roberta_logits = roberta_model(**roberta_inputs).logits
        roberta_probs = F.softmax(roberta_logits, dim=-1).squeeze()
        
    weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
    
    ranked_indices = torch.argsort(weighted_probs, descending=True)
    top_3_predictions = [id2label[idx.item()] for idx in ranked_indices[:3]]
    
    ap3 = calculate_ap3(actual_answer, top_3_predictions)
    ap_scores.append(ap3)

final_map3 = np.mean(ap_scores)
print(f"{final_map3:.4f}")

0.3400
